# MFB 545 · Day 3 · Live credit scoring

**What you will do, in 30 minutes:** load a real consumer-credit dataset (1,000 loans, 300 of them in default), build a score with a logistic regression, measure its quality (AUC, Gini), then choose an **acceptance threshold** and see what it does to the portfolio’s result.

**How to use it.** Run each code cell in order: click it, then ▶ (or `Shift + Enter`). No programming is asked of you: read the results and answer the questions **on the course site** (section 3.2).

*Data: Hofmann, H. (1994), “Statlog (German Credit Data)”, UCI Machine Learning Repository. Amounts in Deutsche Mark of the time.*

## 1 · Load the data

In [ ]:
import pandas as pd, numpy as np
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
columns = ["checking","duration_months","history","purpose","amount","savings","employment","instalment_rate",
           "status_sex","guarantors","residence","property","age","other_credits","housing","n_credits",
           "job","dependants","telephone","foreign_worker","class"]
df = pd.read_csv(url, sep=" ", header=None, names=columns)
df["default"] = (df["class"] == 2).astype(int)        # 1 = the loan went bad
print(len(df), "loans ·", df["default"].sum(), "defaults · default rate", round(df["default"].mean()*100, 1), "%")
df.head()

## 2 · One variable, by hand

The codes in `checking`: **A11** negative balance · **A12** 0 to 200 DM · **A13** 200 DM or more · **A14** no current account at the bank. Look at the default rate per category before any model: this is what a credit analyst did by hand before scores existed.

In [ ]:
df.groupby("checking")["default"].agg(loans="count", default_rate="mean").round(3)

## 3 · The score’s variables

Seven pieces of information, those of an ordinary credit application: the state of the current account (three 0/1 indicators, the reference being “200 DM or more”), the duration in months, the amount in thousands, low savings (under 100 DM), age, and a poor credit history.

**The split.** The model learns on the **first 700** loans and is measured on the **last 300**, which it has never seen. (Usually the split is random; here it is fixed so that the whole class — and the site’s simulator — gets exactly the same numbers.)

In [ ]:
X = pd.DataFrame({
    "checking_negative": (df.checking == "A11").astype(int),
    "checking_0_200":    (df.checking == "A12").astype(int),
    "no_checking":       (df.checking == "A14").astype(int),
    "duration_months":   df.duration_months,
    "amount_thousands":  df.amount / 1000,
    "low_savings":       (df.savings == "A61").astype(int),
    "age":               df.age,
    "poor_history":      df.history.isin(["A30","A31"]).astype(int),
})
y = df["default"]
X_train, y_train, X_test, y_test = X.iloc[:700], y.iloc[:700], X.iloc[700:], y.iloc[700:]
X.head()

## 4 · The logistic regression

The model computes a **logit** = constant + Σ coefficient × variable, then the probability of default **PD = 1 / (1 + e^(−logit))**. A positive coefficient raises the risk; e^(coefficient) is the factor by which the variable multiplies the *odds* of default.

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(penalty=None, max_iter=5000).fit(X_train, y_train)
coef = pd.DataFrame({"coefficient": model.coef_[0], "factor_on_the_odds": np.exp(model.coef_[0])}, index=X.columns)
print("constant:", round(model.intercept_[0], 4))
coef.round(4)

## 5 · Is the score any good? AUC and Gini

The **AUC** is the probability that a randomly drawn bad payer gets a higher PD than a randomly drawn good payer: 0.5 = a coin toss, 1 = perfect. The **Gini** = 2 × AUC − 1 is the measure banks report. **Write down the AUC on the test sample: the site asks for it.**

In [ ]:
from sklearn.metrics import roc_auc_score
pd_test = model.predict_proba(X_test)[:, 1]
auc_test  = roc_auc_score(y_test, pd_test)
auc_train = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
print("AUC train:", round(auc_train, 3), "· AUC test:", round(auc_test, 3), "· Gini test:", round(2*auc_test - 1, 3))

## 6 · Threshold versus profitability

An applicant is accepted if their PD is **below the threshold**. The site simulator’s assumptions: a repaid loan earns **15 %** of its amount (interest net of funding cost, over the life of the loan); a defaulted loan loses **50 %** of its amount (the loss given default, LGD). Change `margin` and `lgd` and run again: the best threshold moves.

In [ ]:
margin, lgd = 0.15, 0.50
amounts, defaults = df.amount.iloc[700:].values, y_test.values
rows = []
for threshold in np.arange(0.05, 0.75, 0.05).round(2).tolist() + [1.01]:
    acc = pd_test < threshold
    gain = np.where(defaults[acc] == 0, margin * amounts[acc], -lgd * amounts[acc]).sum()
    rows.append({"threshold": threshold, "accepted": int(acc.sum()), "defaults": int(defaults[acc].sum()),
                 "default_rate_%": round(100 * defaults[acc].mean(), 1) if acc.any() else 0, "result_DM": round(gain)})
table = pd.DataFrame(rows); table

In [ ]:
import matplotlib.pyplot as plt
thresholds = np.arange(0.01, 0.80, 0.01)
res = [np.where(defaults[pd_test < t] == 0, margin*amounts[pd_test < t], -lgd*amounts[pd_test < t]).sum() for t in thresholds]
plt.figure(figsize=(8, 4)); plt.plot(thresholds*100, res); plt.axhline(0, color="grey", lw=.8)
plt.axvline(100*margin/(margin+lgd), ls="--", color="red", label="theoretical threshold = margin / (margin + LGD)")
plt.xlabel("Acceptance threshold: maximum PD (%)"); plt.ylabel("Portfolio result (DM)"); plt.legend(); plt.show()
print("Best threshold:", round(thresholds[int(np.argmax(res))]*100), "% · result", round(max(res)), "DM")

## 7 · What else the dataset contains

It contains **sex and marital status** (`status_sex`: A91 male divorced/separated · A92 female divorced/separated/married · A93 male single · A94 male married/widowed) and **foreign nationality** (`foreign_worker`: A201 yes · A202 no). The model above does not use them. Look at the default rates per group, then answer on the site: *would you use them if they improved the AUC?* In the EU and Switzerland the answer is no (non-discrimination, data protection; and the AI Act classes credit scoring of individuals as a **high-risk** AI system). But a permitted variable can reproduce a forbidden one: that is the **proxy variable** problem.

In [ ]:
for col in ["status_sex", "foreign_worker"]:
    print(df.groupby(col)["default"].agg(loans="count", default_rate="mean").round(3), "\n")

## To copy back to the site (section 3.2)

1. The **test AUC** (cell 5).
2. The variable with the highest **factor on the odds**, and what it means in one sentence (cell 4).
3. The **best threshold** and its result (cell 6) — then redo it in the page’s simulator, with your own assumptions.